# Imports

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [2]:
keras.mixed_precision.set_global_policy('mixed_float16')

In [3]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm
import transformers

c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
# class Tokenizer(transformers.GPT2Tokenizer):

#     def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
#         if token_ids_1 is None:
#             return [self.bos_token_id, *token_ids_0, self.eos_token_id]

#         return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [6]:
tokenizer: transformers.PreTrainedTokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

In [7]:
assert isinstance(tokenizer.eos_token_id, int)
pad_token_id: int = tokenizer.eos_token_id + 1

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [8]:
df = pd.read_csv("../../data/rnn_seq2seq/rus-eng.tsv", sep="\t", usecols=[1, 3], names=["ru", "en"])

In [9]:
import pickle

In [10]:
TOKENIZE_NOW = False

In [11]:
if TOKENIZE_NOW:
    tokens_ru = [tokenizer.encode(text) for text in df["ru"]]
    with open("../../data/rnn_seq2seq/ru.pkl", "wb") as f:
        pickle.dump(tokens_ru, f, protocol=pickle.HIGHEST_PROTOCOL)

In [12]:
if TOKENIZE_NOW:
    tokens_en = [tokenizer.encode(text) for text in df["en"]]
    with open("../../data/rnn_seq2seq/en.pkl", "wb") as f:
            pickle.dump(tokens_en, f, protocol=pickle.HIGHEST_PROTOCOL)

In [13]:
with open("../../data/rnn_seq2seq/ru.pkl", "rb") as f:
    tokens_ru = pickle.load(f)

with open("../../data/rnn_seq2seq/en.pkl", "rb") as f:
    tokens_en = pickle.load(f)

In [14]:
tokens_ru_len = [len(tokens) for tokens in tokens_ru]
tokens_en_len = [len(tokens) for tokens in tokens_en]

In [15]:
class Seq2SeqDataset(keras.utils.PyDataset):
    def __init__(
        self,
        input_sequences: list[list[int]],
        max_seq_length_input: int,
        output_sequences: list[list[int]],
        max_seq_length_output: int,
        tokenizer: transformers.PreTrainedTokenizer,
        pad_token_id: int,
        batch_size: int = 128,
        shuffle: bool = True,
        rng: np.random.Generator | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)

        if rng is None:
            rng = np.random.default_rng()

        self.input = input_sequences
        self.max_seq_lenth_input = max_seq_length_input
        self.output = output_sequences
        self.max_seq_lenth_output = max_seq_length_output
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = rng
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id
        self.indices = np.arange(len(input_sequences))

        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]

        encoder_input_tokens = [self.input[i] for i in batch_indices]
        decoder_input_tokens = [
            [tokenizer.bos_token_id] + self.output[i] for i in batch_indices
        ]
        decoder_output_tokens = [
            self.output[i] + [tokenizer.eos_token_id] for i in batch_indices
        ]

        encoder_input = keras.utils.pad_sequences(
            encoder_input_tokens,
            maxlen=self.max_seq_lenth_input,
            padding="post",
            value=pad_token_id,
        )
        decoder_input = keras.utils.pad_sequences(
            decoder_input_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=pad_token_id,
        )
        decoder_output = keras.utils.pad_sequences(
            decoder_output_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=pad_token_id,
        )

        mask = (decoder_output != pad_token_id).astype(np.float32)

        return (encoder_input, decoder_input), decoder_output, mask

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [16]:
train_ds = Seq2SeqDataset(tokens_ru, 67, tokens_en, 18, tokenizer, pad_token_id, batch_size=128)

In [17]:
val_translations = [
    ("Привет, как дела?", "Hi, how are you?"),
    ("Я люблю читать книги.", "I love reading books."),
    ("Сегодня хорошая погода.", "The weather is nice today."),
    ("Он не пришёл на встречу.", "He didn't come to the meeting."),
    ("Мы поедем в Москву завтра.", "We will go to Moscow tomorrow."),
    ("Она готовит ужин на кухне.", "She is cooking dinner in the kitchen."),
    ("Мне нравится этот фильм.", "I like this movie."),
    ("Дети играют во дворе.", "The children are playing in the yard."),
    ("Я не знаю, что делать.", "I don't know what to do."),
    ("Этот дом был построен в прошлом году.", "This house was built last year."),
]

In [18]:
val_ru_tokens = []
val_en_tokens = []

for ru, en in val_translations:
    val_ru_tokens.append(tokenizer.encode(ru))
    val_en_tokens.append(tokenizer.encode(en))

In [19]:
val_ds = Seq2SeqDataset(val_ru_tokens, 67, val_en_tokens, 18, tokenizer, pad_token_id, batch_size=10)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [20]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    rnn_type: type[keras.layers.Layer] = keras.layers.LSTM,
) -> keras.Model:
    enc_inputs = keras.Input(shape=(None,), dtype="int32", name="encoder_inputs")
    dec_inputs = keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")

    enc_emb = keras.layers.Embedding(n_tokens, units, name="encoder_embedding")
    dec_emb = keras.layers.Embedding(n_tokens, units, name="decoder_embedding")

    x = enc_emb(enc_inputs)
    enc_outputs = None
    enc_states = None

    for i in range(n_stacks):
        is_last = i == n_stacks - 1
        layer_name = f"encoder_{i}"

        if bidirectional and is_last:
            bi = keras.layers.Bidirectional(
                rnn_type(units=units, return_sequences=True, return_state=True),
                name=layer_name,
            )
            enc_outputs, *states = bi(x)

            enc_outputs = keras.layers.Dense(units, name="enc_output_proj")(enc_outputs)

            n_dir = len(states) // 2
            enc_states = [
                keras.layers.Dense(
                    units,
                    activation="tanh",
                    name=f"enc_state_proj_{i}_{j}",
                )(keras.layers.Concatenate()([states[j], states[n_dir + j]]))
                for j in range(n_dir)
            ]
        else:
            rnn = rnn_type(
                units=units, return_sequences=True, return_state=True, name=layer_name
            )
            enc_outputs, *enc_states = rnn(x)

    y = dec_emb(dec_inputs)
    dec_outputs = None

    for i in range(n_stacks):
        rnn = rnn_type(
            units=units, return_sequences=True, return_state=True, name=f"decoder_{i}"
        )
        if i == 0:
            dec_outputs, *dec_states = rnn(y, initial_state=enc_states)
        else:
            dec_outputs, *dec_states = rnn(y)

    context = keras.layers.AdditiveAttention(name="attention")(
        [dec_outputs, enc_outputs]
    )

    combined = keras.layers.Concatenate(name="concat_context")([dec_outputs, context])

    logits = keras.layers.Dense(n_labels, dtype="float32", name="output")(combined)

    return keras.Model([enc_inputs, dec_inputs], logits, name=name)

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

In [21]:
model = get_model(
    units=256,
    n_tokens=tokenizer.vocab_size + 1,
    n_labels=tokenizer.vocab_size + 1,
    n_stacks=1,
    bidirectional=False,
    name='seq2seq_lstm',
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=['accuracy'],
)

In [22]:
model.summary()

Model: "seq2seq_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 256) │ 12,866,048 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │ 12,866,048 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0 (LSTM)    │ [(None, None,     │    525,312 │ encoder_embeddin… │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_0 (LSTM)    │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_0[0][1],  │
│                     │ 256), (None,      │            │ encoder_0[0][2]   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 256) │        256 │ decoder_0[0][0],  │
│ (AdditiveAttention) │                   │            │ encoder_0[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_context      │ (None, None, 512) │          0 │ decoder_0[0][0],  │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │ 25,782,354 │ concat_context[0… │
│                     │ 50258)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,565,330 (200.52 MB)

 Trainable params: 52,565,330 (200.52 MB)

 Non-trainable params: 0 (0.00 B)

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

In [23]:
from pathlib import Path

In [24]:
LOGS_DIR = Path('../../logs/rnn_seq2seq')
CHECKPOINT_DIR = Path('../../model_checkpoints/rnn_seq2seq')

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [25]:
from datetime import datetime

In [26]:
from torch.utils.tensorboard import SummaryWriter


class PyTorchTensorBoard(keras.callbacks.Callback):
    def __init__(self, log_dir):
        super().__init__()
        self.writer = SummaryWriter(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        if logs:
            for key, value in logs.items():
                self.writer.add_scalar(key, value, epoch)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_train_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="accuracy",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_val_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),  
    PyTorchTensorBoard(LOGS_DIR),
]

In [28]:
model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

Epoch 1/5


c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


2024/6487 ━━━━━━━━━━━━━━━━━━━━ 19:33 263ms/step - accuracy: 0.3119 - loss: 2.2102

KeyboardInterrupt: 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [ ]:
def build_inference_models(model, cell_type=keras.layers.LSTM):
    n_states = 2 if cell_type is keras.layers.LSTM else 1

    enc_emb = model.get_layer("encoder_embedding")
    dec_emb = model.get_layer("decoder_embedding")
    enc_lstm = model.get_layer("encoder_0")
    dec_lstm = model.get_layer("decoder_0")
    attention = model.get_layer("attention")
    out_dense = model.get_layer("output")
    units = enc_lstm.units

    enc_in = keras.Input(shape=(None,), dtype="int32", name="enc_in_inf")
    x = enc_emb(enc_in)
    enc_out, *enc_states = enc_lstm(x)
    encoder_inf = keras.Model(enc_in, [enc_out] + enc_states, name="encoder_inf")

    dec_in = keras.Input(shape=(1,), dtype="int32", name="dec_in_inf")
    enc_out_in = keras.Input(shape=(None, units), name="enc_out_in")
    state_ins = [
        keras.Input(shape=(units,), name=f"state_in_{j}") for j in range(n_states)
    ]

    y = dec_emb(dec_in)
    y, *new_states = dec_lstm(y, initial_state=state_ins)

    context = attention([y, enc_out_in])
    combined = keras.layers.Concatenate()([y, context])
    logits = out_dense(combined)

    decoder_inf = keras.Model(
        [dec_in, enc_out_in] + state_ins,
        [logits] + new_states,
        name="decoder_inf",
    )

    return encoder_inf, decoder_inf

In [ ]:
encoder_inf, decoder_inf = build_inference_models(model, cell_type=keras.layers.LSTM)

In [ ]:
def translate(text, tokenizer, encoder_inf, decoder_inf, max_len=50):
    bos = tokenizer.bos_token_id
    eos = tokenizer.eos_token_id
    pad = tokenizer.pad_token_id

    enc_ids = tokenizer(text, add_special_tokens=False, return_tensors="np")[
        "input_ids"
    ]
    outputs = encoder_inf.predict(enc_ids, verbose=0)
    enc_out = outputs[0]
    states = outputs[1:]

    tokens = []
    next_id = bos

    for _ in range(max_len):
        dec_in = np.array([[next_id]], dtype="int32")
        outputs = decoder_inf.predict([dec_in, enc_out] + states, verbose=0)
        logits, *states = outputs
        next_id = int(np.argmax(logits[0, -1, :]))

        if next_id == eos or next_id == pad:
            break
        tokens.append(next_id)

    return tokenizer.decode(tokens, skip_special_tokens=True)

In [ ]:
for ru, en in val_translations:
    pred = translate(ru, tokenizer, encoder_inf, decoder_inf)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Do you know how to get a haircut?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: This weather is good today.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We're going to go to school.
------------------------------------------------------------
RU  : Она готовит ужин на кухне.
REF : She is cooking dinner in the kitchen.
PRED: She is preparing dinner for dinner.
------------------------------------------------------------
RU  : Мне нравится этот фильм.
REF : 

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10, initial_epoch=5, callbacks=callbacks)

Epoch 6/10


c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1228s 189ms/step - accuracy: 0.7627 - loss: 0.5148 - val_accuracy: 0.8846 - val_loss: 0.1729
Epoch 7/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1238s 191ms/step - accuracy: 0.7761 - loss: 0.4764 - val_accuracy: 0.8590 - val_loss: 0.1595
Epoch 8/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1235s 190ms/step - accuracy: 0.7868 - loss: 0.4463 - val_accuracy: 0.8718 - val_loss: 0.1512
Epoch 9/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1207s 186ms/step - accuracy: 0.7956 - loss: 0.4223 - val_accuracy: 0.8718 - val_loss: 0.1631
Epoch 10/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1242s 191ms/step - accuracy: 0.8031 - loss: 0.4022 - val_accuracy: 0.8846 - val_loss: 0.1495


In [ ]:
encoder_inf, decoder_inf = build_inference_models(model, cell_type=keras.layers.LSTM)

In [ ]:
for ru, en in val_translations:
    pred = translate(ru, tokenizer, encoder_inf, decoder_inf)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Do you know how to get a radio?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: Today is good weather.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We'll go to school.
------------------------------------------------------------
RU  : Она готовит ужин на кухне.
REF : She is cooking dinner in the kitchen.
PRED: She is preparing for dinner.
------------------------------------------------------------
RU  : Мне нравится этот фильм.
REF : I like this movie.
PRED